# NB11 — Splits (SPEC T1)

Builds the dev and fold splits that every later notebook uses. **CPU only, no GPU.**

- **Extra-dev:** 20% of Iqra_Extra_IS26, speaker-disjoint if a speaker field exists, else sentence-disjoint (canonical `phoneme_ref`). The rest goes to fold A and fold B the same way. Seed 13.
- **Iqra dev slice:** ~1 h of Iqra_train `dev` (verifier negatives). No TTS slice (decision 9).
- Writes `splits.json`, `truth_extra_dev.csv` and, only if IDs had to be made safe, `id_map.json`.
- Never touches QuranMB.v2 or IqraEval_Test_GT.

**Run settings:** Accelerator *None*, Internet *On*. Add data: the `iqra-cache` dataset. Secrets: `HF_TOKEN` (gated Iqra_Extra_IS26); `GITHUB_TOKEN` only if the repo is private.

**After the run:** Save Version (commit), then *Create Dataset* from the output, named `iqra-splits`. Paste back everything printed by the last cell (**PASTE BACK**).


In [ ]:
# ---- Shared install cell (identical in nb11-nb16) --------------------------------
# pip install git+... does not fetch submodules, and the official scorer is a pinned
# submodule, so clone recursively and install from the clone (SPEC decision 21).
import os, subprocess, sys
REPO_URL = "https://github.com/markeet04/Quran_phenome.git"
REPO_DIR = "/tmp/Quran_phenome"          # outside /kaggle/working: not saved as output
BRANCH = "main"

token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")   # only if the repo is private
except Exception:
    pass
url = REPO_URL.replace("https://", f"https://x-access-token:{token}@") if token else REPO_URL

def _git(*args):
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    if r.returncode:
        msg = r.stderr if not token else r.stderr.replace(token, "***")
        raise RuntimeError(f"git {args[0]} failed: {msg}")
    return r.stdout.strip()

if not os.path.isdir(REPO_DIR):
    _git("clone", "--recursive", "-q", "-b", BRANCH, url, REPO_DIR)
else:   # a re-run in the same session: move the existing clone to the latest commit
    _git("-C", REPO_DIR, "fetch", "-q", url, BRANCH)
    _git("-C", REPO_DIR, "reset", "-q", "--hard", "FETCH_HEAD")
_git("-C", REPO_DIR, "submodule", "update", "--init", "--recursive", "-q")
REPO_COMMIT = _git("-C", REPO_DIR, "rev-parse", "HEAD")

# No datasets pin: Kaggle now runs Python 3.13, where datasets 2.21's dependencies break.
# Audio is decoded by iqra.data (soundfile on the raw bytes), so any datasets version works.
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR],
                   capture_output=True, text=True)
if r.returncode:
    raise RuntimeError("pip install failed:\n" + r.stdout[-1500:] + "\n" + r.stderr[-1500:])
sys.path.insert(0, f"{REPO_DIR}/src")
import warnings
warnings.filterwarnings("ignore")
import iqra.align
import datasets, numpy, pandas
print("python", sys.version.split()[0], "| datasets", datasets.__version__,
      "| numpy", numpy.__version__, "| pandas", pandas.__version__)
print("repo commit:", REPO_COMMIT)
print("official scorer:", iqra.align.mdd_eval_dir())


In [ ]:
# ---- Config and tokens ---------------------------------------------------------------
import json
import numpy as np
import pandas as pd
from datasets import load_dataset, load_from_disk
from iqra import data, scorer
from iqra.config import Config
from iqra.vocab import Vocab

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = HF_TOKEN
except Exception as e:
    raise RuntimeError("HF_TOKEN secret is required for the gated Iqra_Extra_IS26") from e

cfg = Config()
from iqra.config import find_cache_root
if os.path.isdir("/kaggle/input"):
    cfg.paths.cache_root = find_cache_root()
print("iqra-cache:", cfg.paths.cache_root)
OUT = "/kaggle/working/nb11"
os.makedirs(OUT, exist_ok=True)
SEED = cfg.splits.seed
report = {"repo_commit": REPO_COMMIT, "seed": SEED}
vocab = Vocab.load(cfg.paths.vocab)
print("vocab:", len(vocab.tok2id), "tokens | hash", vocab.hash)


In [ ]:
# ---- 1. Real column names of every dataset ---------------------------------------------
train_dd = load_from_disk(cfg.paths.train_arrow)
tts = load_from_disk(cfg.paths.tts_arrow)
tts = tts["train"] if hasattr(tts, "keys") and "train" in tts.keys() else tts
extra = load_dataset(cfg.datasets.extra, split="train", token=HF_TOKEN)

cols = {"Iqra_train": train_dd["train"].column_names, "Iqra_train dev": train_dd["dev"].column_names,
        "Iqra_TTS": tts.column_names, "Iqra_Extra_IS26": extra.column_names}
for name, c in cols.items():
    print(f"{name:16s} {c}")
print("sizes: train", len(train_dd["train"]), "| dev", len(train_dd["dev"]),
      "| TTS", len(tts), "| Extra", len(extra))

spk_col = data.find_column(extra.column_names, data.SPEAKER_COLUMNS)
id_col = data.find_column(extra.column_names, data.ID_COLUMNS)
txt_col = data.find_column(extra.column_names, data.TEXT_COLUMNS)
print("\nExtra speaker field:", spk_col, "| ID field:", id_col, "| text field:", txt_col)
if "speaker" in tts.column_names:
    spk = pd.Series(list(tts["speaker"])).value_counts()
    print("TTS speakers:", dict(spk), "| 'Amer' present:", "Amer" in spk.index)
report["columns"] = cols
report["extra_fields"] = {"speaker": spk_col, "id": id_col, "text": txt_col}


In [ ]:
# ---- 2. Extra: clean rows, safe IDs, group split -------------------------------------
canon = [str(x) if x is not None else "" for x in list(extra["phoneme_ref"])]
said = [str(x) if x is not None else "" for x in list(extra["phoneme_mis"])]
empty = [i for i in range(len(extra)) if not canon[i].strip() or not said[i].strip()]
empty_set = set(empty)
print("rows with empty phoneme_ref/phoneme_mis (excluded):", len(empty))
keep_rows = [i for i in range(len(extra)) if i not in empty_set]

raw_ids = list(extra[id_col]) if id_col else [None] * len(extra)
ids_all, id_map = data.safe_ids(raw_ids, "extra")
print("IDs:", "original" if id_map is None else "replaced with safe IDs (id_map.json)")

spk_values = list(extra[spk_col]) if spk_col else None
groups = [str(spk_values[i]) if spk_col else canon[i] for i in keep_rows]
GROUP_KEY = spk_col or "phoneme_ref (canonical sentence)"
local = data.group_split(groups, cfg.splits.extra_dev_frac, SEED)
data.assert_group_disjoint(local, groups)
split_rows = {k: [keep_rows[i] for i in v] for k, v in local.items()}      # dataset row indices
print("group key:", GROUP_KEY, "| groups:", len(set(groups)))
for k, v in local.items():
    print(f"  {k:9s} {len(v):5d} clips, {len({groups[i] for i in v})} groups")

# Sentence overlap between Extra-dev and the folds. Expected > 0 only for a speaker split;
# the LM corpus filter (decision 14) removes those sentences.
dev_sent = {canon[r] for r in split_rows["extra_dev"]}
fold_sent = {canon[r] for k in ("fold_a", "fold_b") for r in split_rows[k]}
print("Extra-dev sentences also in folds:", len(dev_sent & fold_sent), "of", len(dev_sent))
report["extra_split"] = {"group_key": GROUP_KEY, "excluded_empty_rows": empty,
                         "sizes": {k: len(v) for k, v in split_rows.items()},
                         "groups": {k: len({groups[i] for i in v}) for k, v in local.items()},
                         "dev_sentences_in_folds": len(dev_sent & fold_sent)}


In [ ]:
# ---- 3. Audio: hours per split and identical-audio check ------------------------------
fp, dur = {}, {}
for i, arr, sr in data.iter_audio(extra):          # raw bytes + soundfile, any datasets version
    fp[i] = data.audio_fingerprint(arr)
    dur[i] = len(arr) / sr
owner, cross = {}, 0
for k, rows in split_rows.items():
    for r in rows:
        if fp[r] in owner and owner[fp[r]] != k:
            cross += 1
        owner.setdefault(fp[r], k)
n_rows = sum(len(v) for v in split_rows.values())
dups_within = n_rows - len(owner) - cross
assert cross == 0, f"{cross} identical audio clips shared across splits"
hours = {k: round(sum(dur[r] for r in v) / 3600, 3) for k, v in split_rows.items()}
print("hours:", hours, "| identical audio across splits:", cross, "| within a split:", dups_within)
report["extra_split"].update({"hours": hours, "identical_audio_across": cross,
                              "identical_audio_within": dups_within})


In [ ]:
# ---- 4. Iqra dev slice (~1 h of Iqra_train dev) ---------------------------------------
dev = train_dd["dev"]
dev_dur = [len(arr) / sr for _, arr, sr in data.iter_audio(dev)]
slice_rows, slice_hours = data.duration_slice(dev_dur, cfg.splits.iqra_dev_slice_hours, SEED)
dev_id_col = data.find_column(dev.column_names, data.ID_COLUMNS)
dev_id_values = list(dev[dev_id_col]) if dev_id_col else None
dev_raw_ids = [dev_id_values[r] for r in slice_rows] if dev_id_col else [None] * len(slice_rows)
dev_ids, dev_id_map = data.safe_ids(dev_raw_ids, "iqradev")
print(f"Iqra dev slice: {len(slice_rows)} clips, {slice_hours:.3f} h (of {len(dev)} dev clips, "
      f"{sum(dev_dur) / 3600:.2f} h) | IDs:", "original" if dev_id_map is None else "safe-mapped")
report["iqra_dev_slice"] = {"clips": len(slice_rows), "hours": round(slice_hours, 3)}


In [ ]:
# ---- 5. Share of Quranic sentences in Extra (risk: dev differs from test) ------------
try:
    quran = data.load_tanzil_simple_clean()
    if txt_col:
        texts = list(extra[txt_col])
        flags = data.quranic_flags(texts, quran)
        share = {k: round(float(np.mean([flags[r] for r in v])), 3) for k, v in split_rows.items()}
        share["all"] = round(float(np.mean([flags[r] for r in keep_rows])), 3)
        print("Quranic share (verbatim match against Tanzil simple-clean):", share)
        print("examples, Quranic:", [texts[r] for r in keep_rows if flags[r]][:3])
        print("examples, not Quranic:", [texts[r] for r in keep_rows if not flags[r]][:3])
    else:
        share = "unknown: Extra has no text column"
        print(share)
except Exception as e:
    share = f"unknown: {type(e).__name__}: {e}"
    print("Quranic share", share)
report["quranic_share"] = share


In [ ]:
# ---- 6. Label sanity: vocab coverage and error counts ---------------------------------
phon = vocab.phonemes()
oov = sorted({t for r in keep_rows for s in (canon[r], said[r]) for t in s.split()} - phon)
no_err = {k: sum(canon[r] == said[r] for r in v) for k, v in split_rows.items()}
print("tokens not in model vocab:", oov or "none")
print("clips with no error (phoneme_mis == phoneme_ref):", no_err)
report["label_sanity"] = {"oov_tokens": oov, "clips_without_error": no_err}


In [ ]:
# ---- 7. Write splits.json, truth_extra_dev.csv, id maps --------------------------------
splits = {
    "seed": SEED, "repo_commit": REPO_COMMIT,
    "extra": {
        "dataset": cfg.datasets.extra, "fingerprint": getattr(extra, "_fingerprint", None),
        "n_rows": len(extra), "group_key": GROUP_KEY, "excluded_empty_rows": empty,
        "ids_by_row": ids_all,
        **{k: [ids_all[r] for r in v] for k, v in split_rows.items()},
    },
    "iqra_dev_slice": {
        "dataset": cfg.paths.train_arrow + " [dev]", "fingerprint": getattr(dev, "_fingerprint", None),
        "rows": slice_rows, "ids": dev_ids, "hours": round(slice_hours, 3),
    },
}
data.save_splits(splits, f"{OUT}/splits.json")
truth = data.truth_frame([ids_all[r] for r in split_rows["extra_dev"]],
                         [canon[r] for r in split_rows["extra_dev"]],
                         [said[r] for r in split_rows["extra_dev"]])
truth.to_csv(f"{OUT}/truth_extra_dev.csv", index=False)
maps = {k: m for k, m in (("extra", id_map), ("iqra_dev_slice", dev_id_map)) if m}
if maps:
    with open(f"{OUT}/id_map.json", "w", encoding="utf-8") as f:
        json.dump(maps, f, ensure_ascii=False, indent=1)
print("wrote:", sorted(os.listdir(OUT)))


In [ ]:
# ---- 8. Done-when checks: perfect and null predictors on truth_extra_dev.csv ----------
truth = pd.read_csv(f"{OUT}/truth_extra_dev.csv", keep_default_na=False)
perfect = truth.rename(columns={"Annotation_phn": "Prediction"})[["ID", "Prediction"]]
res_p = scorer.official_score(truth, perfect)
assert res_p["f1"] == 1.0, res_p
null = truth.rename(columns={"Reference_phn": "Prediction"})[["ID", "Prediction"]]
counts = scorer.per_clip_counts(truth, null)
assert int(counts["TR"].sum()) == 0
try:
    res_n = scorer.official_score(truth, null)
    assert res_n["recall"] == 0.0
    null_msg = f"official recall {res_n['recall']} (TR=0)"
except scorer.ScorerUndefined:
    null_msg = "TR=0 and FR=0: official script divides by zero (recall 0 from counts)"
print("perfect predictor: F1", res_p["f1"], "| null predictor:", null_msg)
report["checks"] = {"disjoint": True, "perfect_f1": res_p["f1"], "null": null_msg,
                    "extra_dev_reference_errors": int(res_p["TR"])}
with open(f"{OUT}/nb11_report.json", "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=1, default=str)


In [ ]:
# ---- PASTE BACK ----------------------------------------------------------------------
print(json.dumps({k: report[k] for k in ("repo_commit", "extra_fields", "extra_split",
                  "iqra_dev_slice", "quranic_share", "label_sanity", "checks")},
                 ensure_ascii=False, indent=1, default=str))
print("columns:", json.dumps(report["columns"], ensure_ascii=False))
